# Analysis notebook for PiC_UVnudge runs
## Set up
### Packages

In [ ]:
import numpy as np
import xarray as xr
import xesmf as xe
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import pandas as pd
import scipy
from scipy import stats, interpolate
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.mathtext import _mathtext as mathtext
import matplotlib.ticker as mticker
from matplotlib import gridspec, animation
import matplotlib.path as mpath
import matplotlib.colors as colors
import matplotlib.dates as mdates
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import warnings
warnings.simplefilter('ignore', UserWarning)
warnings.filterwarnings('ignore')
import datetime as dt
from datetime import timedelta
from cmcrameri import cm
from Processing_functions import lats, arclats, lons
import jinja2
import cftime
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client
from functools import partial
from collections import defaultdict
import os
import copy

In [ ]:
## Plot types to make - CHANGE
# 0: Weighted spatial mean or Arctic sea ice area
# 1: Spatial maximum (AMOC)
# 2: Leave alone (doing spatial map or sea ice concentration)
# 3: Zonal
plots = {
    'amoc': [False, 1],
    'lin': [False, 0],
    'mtrd': [True, 0],
}

## Categorical plot type - DO NOT CHANGE
plot_types = {
    'line': [False, []],
    'mtrd': [False, []]
}

# Set up plot_types based on plots
for pl, att in plots.items():
    if att[0]:
        if pl == 'mtrd':
            plot_types['mtrd'][0] = True
            plot_types['mtrd'][1].append(pl)
        elif att[1] <= 1:
            plot_types['line'][0] = True
            plot_types['line'][1].append(pl)

# Spatial domain - CHANGE s_domain & t_domain only
s_domain = True # True: Global, False: Arctic
r_domain = False # True: use Arctic regional domain, False: pan-Arctic average
a_domain = False # True: 50-90, False: 70-90
o_domain =  True # True: 20-60N max at each lat, False: 33-55N max
subset_time = False # True: slice time in MasterDS, False: Need all data post-1950 for XX-yr trends
t_domain = 1950 # start year
trd_length = 20 # trend length

## Time averaging type - CHANGE
time_avg = 4  # 0: Monthly, 1: Yearly, 2: Seasonal, 3: All data, 4: Timeseries

## Ensemble mean or All members - CHANGE
ens_type = 0   # 0: All_members, 1: Mean

# Variables - CHANGE
comp = 'ocn' # compset
freq = 0 # 0: monthly, 1: daily
var_ind = 0

In [ ]:
# DO NOT CHANGE
var_list = {'atm': ['TREFHT'],
            'ice': ['aice','hi'],
            'ocn': ['MOC']}
var_ext = {0: '', 1: '_d'}
var = var_list[comp][var_ind]+var_ext[freq]

In [ ]:
## Test numbers - DO NOT CHANGE
tst_nums = np.arange(1,4)

## Test names
# O: LENS ensemble
# 1: PiC_UVnudge ensemble
# 2: HIST_UVnudge ensemble
# 3: observations
# attribute structure: [use dataset, dataset type, line color, line style, zorder]
# CHANGE ONLY - use dataset
use_era5 = var in ['aice','TREFHT']
ds_names = {
    'LENS2 piControl': [True, 0],
    'LENS2 HIST-SSP370': [True, 0],
    'GHG2': [True, 0],
    'PiC_UVnudge_LM2006': [True, 1],
    'HIST_UVnudge_LM': [True, 2],
    'GHG_UVnudge_LM': [True, 1],
    'ERA5': [use_era5, 3],
    'HadCRUT5': [var == 'TREFHT' and plots['mtrd'][0], 3],
    'BEST': [var == 'TREFHT' and plots['mtrd'][0], 3],
    'GISTEMP': [var == 'TREFHT' and (plots['mtrd'][0] or plots['lin'][0]), 3],
    'NSIDC': [var == 'aice' and (plots['mtrd'][0] or plots['lin'][0]), 3],
    'OSISAF': [var == 'aice' and (plots['mtrd'][0] or plots['lin'][0]), 3],
    'RAPID': [var == 'MOC' and (plots['mtrd'][0] or plots['amoc'][0]), 3]
}
vercompres = 'b.e21.B1850cmip6.f09_g17.'
cesm2le = 'b.e21.BHISTsmbb.f09_g17.'
cesm2piC = 'b.e21.B1850.f09_g17.'
le_names = {'LENS2 piControl': cesm2piC+'CMIP6-piControl.001',
            'LENS2 HIST-SSP370': cesm2le+'LE-smbb.*',
            'GHG2': vercompres+'SF2-GHG.*'}

## Paper names - DO NOT CHANGE
ds_paper_names = {
    'LENS2 HIST-SSP370': 'HIST-SSP370-control',
    'GHG2': 'GHG-control',
    'LENS2 piControl': 'PI-control',
    'PiC_UVnudge_LM2006': 'PI+winds',
    'ERA5': 'OBS (ERA5)',
    'GISTEMP': 'OBS (GISTEMP)',
    'HadCRUT5': 'OBS (HadCRUT5)',
    'BEST': 'OBS (BEST)',
    'NSIDC': 'OBS (NSIDC)',
    'OSISAF': 'OBS (OSISAF)',
    'RAPID': 'OBS (RAPID)',
    'HIST_UVnudge_LM': 'HIST+winds',
    'GHG_UVnudge_LM': 'GHG+winds',
}


## Filepaths - DO NOT CHANGE
path_to_work = '/glade/work/glydia/'
path_to_lensdata = path_to_work+'processed_CESM2_LENS_data/longer_recent/'
path_to_expdata = path_to_work+'Arctic_controls_processed_data/'
path_to_plotdata  = path_to_expdata+'recent_plotting_data/'

# Extensions - DO NOT CHANGE
h_ext = {'atm': ['.h0.'],
       'ice': ['.h.','.h1.'],
       'ocn': ['.h.']}
yr_extn = {False: [".195001-202412.",".19500101-20241231."],
           True: [".*.",".05010101-05741231."]}
vert_lev = {'atm': [False],
            'ice': [False,False],
            'ocn': [False]}
file_bool = not vert_lev[comp][var_ind] and freq == 0
file_ext = {True: 'nc', False: 'zarr'}

In [5]:
########################## DO NOT CHANGE ANYTHING BELOW THIS LINE #############################

In [6]:
%%time
    
## Select plot type
time_str_list = {0: 'month', 1: 'year', 2: 'season', 3: 'all', 4: 'timeseries'}
time_outstr = time_str_list[time_avg]

## Select ensemble type
ens_str_list = {0: 'All_members', 1: 'Mean'}
ens_str = ens_str_list[ens_type]

## Select time and spatial domain strings
s_domain = True if ((var == 'RESTOM' and plots['lin'][0]) or comp == 'ocn') else s_domain # Make sure TOA is global domain
sd_str_list = {True: 'Global', False: 'Arctic'}
sd_str = sd_str_list[s_domain]

rd_str_list = {True: 'Reg', False: ''}
rd_str = rd_str_list[r_domain]

od_str_list = {True: 'Lat', False: ''}
od_str = od_str_list[o_domain]

td_str = str(t_domain)
t_end = 2024

CPU times: user 7 µs, sys: 1 µs, total: 8 µs
Wall time: 11.7 µs


In [7]:
## Print Script Configurations
# Plot types
print('Plot types: ')
catcount = 0

for cat, att in plot_types.items():
    if att[0]:
        print('   '+cat)
        catcount += 1
        for pl in att[1]:
            print('      '+pl)

# Variable
if var == 'Z3':
    print('Variable: Z3, U, V')
else:
    print('Variable(s): '+var)

# Datasets
print('Datasets:')
for dsname, attr in ds_names.items():
    if attr[0]:
        print('   '+dsname)

# Spatial domain
print('Spatial domain: '+sd_str)

# Regional division
print('Regional domains: '+('Yes' if r_domain else 'No'))

# Ocean latitude
print('Ocean domain: '+('20-60N' if o_domain else '33-55N'))

# Subset time
print('Subsetting time: '+('Yes' if subset_time else 'No'))
if not subset_time:
    print('Trend length: '+str(trd_length)+' years')

# Time domain
print('Time domain: '+td_str+'-'+str(t_end))

# Time averaging
print('Time averaging: '+time_outstr)

# Ensemble
print('Ensemble averaging: '+ens_str)

Plot types: 
   mtrd
      mtrd
Variable(s): MOC
Datasets:
   LENS2 piControl
   LENS2 HIST-SSP370
   GHG2
   PiC_UVnudge_LM2006
   HIST_UVnudge_LM
   GHG_UVnudge_LM
   RAPID
Spatial domain: Global
Regional domains: No
Ocean domain: 20-60N
Subsetting time: No
Trend length: 20 years
Time domain: 1950-2024
Time averaging: timeseries
Ensemble averaging: All_members


In [8]:
## Check number of different averaging types
if catcount > 1:
    raise SystemExit("ERROR: More than one averaging type fed into CreateMasterDS()")

### Cluster

In [9]:
cluster = PBSCluster(cores    = 1,
                     memory   = '35GiB',
                     queue    = 'casper',
                     walltime = '12:00:00',
                     account  = 'UCUB0137',
                     log_directory = '/glade/u/home/glydia/python/logs/',
                     name='PiC_HIST_UVnudge_process_'+var)
cluster.scale(4*9)
client = Client(cluster)

In [10]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.116:36131,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


### Custom functions

#### pre_process

In [11]:
def pre_process(da):
    da = da.assign_coords(time=pd.date_range('1950-01-01',str(t_end)+'-12-01',freq='MS'))
    return da

#### pre_processlm

In [12]:
def pre_processlm(da):
    da = da.assign_coords(time=pd.date_range('1990-01-01','2014-12-01',freq='MS'))
    return da

#### LoadData

In [13]:
def LoadData(casename, set_type, varname):
    # Condition for only using 501-574 from LENS2 piControl, i.e. when plotting Z3
    cond_lens = var != 'Z3' # True means use 50 slices, False means use 501-574 only
    
    # Create file name
    if set_type == 0:
        # If cond_lens true, select all netcdf files with *
        # Else, select zarr file with 1950-2024 time string
        filename = le_names[casename]+h_ext[comp][freq]+varname+yr_extn[cond_lens][freq]+file_ext[file_bool]
        totalpath = path_to_lensdata+filename
    elif set_type == 3:
        filename = casename+h_ext[comp][freq]+varname+yr_extn[False][freq]+file_ext[file_bool]
        totalpath = path_to_work+'processed_'+casename+'_data/'+filename
    elif set_type == 2:
        filename = cesm2le+casename+h_ext[comp][freq]+varname+yr_extn[False][freq]+file_ext[file_bool]
        totalpath = path_to_expdata+'processed_'+casename+'_data/'+filename
    else:
        filename = vercompres+casename+h_ext[comp][freq]+varname+yr_extn[False][freq]+file_ext[file_bool]
        totalpath = path_to_expdata+'processed_'+casename+'_data/'+filename

    # Load if piControl and using slices
    if set_type == 0 and cond_lens: 
        if casename == 'CESM2-lessmelt HIST':
            data = xr.open_mfdataset(totalpath, combine='nested',concat_dim='slice',preprocess=pre_processlm)
        else:
            data = xr.open_mfdataset(totalpath, combine='nested',concat_dim='slice',preprocess=pre_process)  
    
    # Load if netCDF and not piControl
    elif file_bool:
        data = xr.open_dataset(totalpath, chunks={'time':12})  

    # Load if Zarr
    else:
        data = xr.open_zarr(totalpath, group=varname,  chunks={'time':12})

    return data

In [14]:
# Set spatial domain slicing
slice_ocn = dict(lat_aux_grid=slice(33,55),moc_z=slice(800*100,2200*100), moc_comp=0,transport_reg=1)
slice_ocnN = dict(lat_aux_grid=slice(20,60),moc_z=slice(800*100,2200*100), moc_comp=0,transport_reg=1)
slice_icemod = dict(nj=slice(250,385))
slice_iceobs = dict(lat=slice(50,90))
slice_atmwei = dict(lat=slice(70,90))
slice_atmspt = dict(lat=slice(50,90))
slice_time = dict(time=slice(td_str+'-01-01',str(t_end)+'-12-31'))
slice_rdom = {'HA': dict(lat=slice(80,90)),
              'G': dict(lat=slice(66.5,90),lon=slice(-70,0)),
              'AA': dict(lat=slice(66.5,90),lon=slice(0,60)),
              'WS': dict(lat=slice(66.5,90),lon=slice(60,120)),
              'ES': dict(lat=slice(66.5,90),lon=slice(120,180)),
              'PA': dict(lat=slice(66.5,90),lon=slice(-180,-70))}

#### CreateMasterDS

In [15]:
def CreateMasterDS(varname):
    ds_load_list = []
    pd_time = pd.date_range('1950-01-01',str(t_end)+'-12-01',freq='MS')
    pd_stime = pd.date_range('1950-01-01','2023-12-01',freq='MS')
    pd_lmtime = pd.date_range('1990-01-01','2014-12-01',freq='MS')

    # Find which plots will be made
    funcstr = None
    for plotname, attrs in plots.items():
        # If plot will be made
        if attrs[0]:
            # If weighted spatial average or Arctic sea ice area
            if attrs[1] == 0:
                if varname == 'aice':
                    funcstr = 'S'
                elif varname == 'MOC':
                    funcstr = 'M'
                else:
                    funcstr = 'W'
                break
            # Elif spatial maximum
            elif attrs[1] == 1:
                funcstr = 'M'
                break
            # Elif zonal average
            elif attrs[1] == 3:
                funcstr = 'Z'
                break

    print('Figure out processing calculations to do')

    ## Load all datasets
    # Load each dataset
    for dsname, attrs in ds_names.items():
        # If dataset is going to be plotted
        if attrs[0]:
            print(dsname)
            # If dataset is ERA5
            if dsname == 'ERA5':
                # If using U or V variable, use Target_U & Target_V from PiC_UVnudge
                if varname == 'U' or varname == 'V':
                    varname = 'Target_'+varname
                    ds_load = LoadData('PiC_UVnudge', 1, varname)
                    ds_load = ds_load.mean('ensemble_member')
                else:
                    ds_load = LoadData(dsname, attrs[1], varname)
                    ds_load = CalcGridArea(ds_load)
            else:
                ds_load = LoadData(dsname, attrs[1], varname)
                if dsname == 'DA_SIC_RECON':
                    ds_load = ds_load.rename({'gridarea_sqkm':'area'})
    
            # Reduce to dataarray
            ds_load = ds_load[varname]
        
            # Rename dataarray name to be casename
            ds_load = ds_load.rename(dsname)
            
            print('  Initial data loading complete')

            # Make all datasets have the same time axis
            if dsname == 'CESM2-lessmelt HIST':
                ds_load = ds_load.assign_coords(time=pd_lmtime)
            elif dsname == 'DA_SIC_RECON':
                ds_load = ds_load.assign_coords(time=pd_stime)
            elif dsname != 'NSIDC' and dsname != 'CERES' and dsname != 'OSISAF' and dsname != 'RAPID':
                ds_load = ds_load.assign_coords(time=pd_time)
            else:
                stime = ds_load.time[0]
                etime = ds_load.time[-1]
                pd_ctime = pd.date_range(str(stime.dt.year.values)+'-'+str(stime.dt.month.values).zfill(2)+'-01',
                                        str(etime.dt.year.values)+'-'+str(etime.dt.month.values).zfill(2)+'-01',
                                        freq='MS')
                ds_load = ds_load.assign_coords(time=pd_ctime)

            print('  Fixed time dimension')
                

            ## Do weighted averages, ensemble means, sea ice area calculations, and area maximums
            # Do spatial domain slicing
            # ICE
            if comp == 'ice' and dsname != 'NSIDC' and dsname != 'OSISAF':
                ds_load = ds_load.loc[slice_iceobs] if attrs[1] == 3 else ds_load.loc[slice_icemod]
            # OCEAN
            elif comp == 'ocn' and attrs[1] != 3:
                ds_load = ds_load.loc[slice_ocnN].drop('moc_components') if o_domain else ds_load.loc[slice_ocn].drop('moc_components')
            # ATMOSPHERE
            elif comp == 'atm' and (not s_domain and not r_domain):
                # Make sure all atm variables have same lat/lon values
                ds_load = ds_load.assign_coords({'lat':lats,'lon':lons}) if attrs[1] != 3 else ds_load
                ds_load = ds_load.loc[slice_atmspt] if a_domain else ds_load.loc[slice_atmwei]
            
                
            print('  Sliced data')
                    
            # Sort processing by variable and graph type
            # Doing spatial plot or SIC
            if funcstr == None or funcstr == 'Z':
                print('  No proceesing, spatial')
                # If ERA5, rename lat/lon
                if dsname == 'ERA5':
                    ds_load = ds_load.rename({'lat':'latE', 'lon':'lonE'})
                    if varname == 'Z3':
                        ds_load = ds_load.rename({'lev':'plev'})

                if funcstr == 'Z':
                    print('  Calculating zonal average')
                    if dsname == 'ERA5':
                        ds_load = ds_load.mean('lonE', skipna=True)
                    else:
                        ds_load = ds_load.mean('lon', skipna=True)
                else:
                    print('  No proceesing, spatial')
                
                    
            # Doing SIA plot(s)
            elif funcstr == 'S':
                print('  Calculating sea ice extent')
                if dsname != 'NSIDC' and dsname != 'OSISAF':
                    ds_load = CalcSIE(ds_load, attrs[1])
            # Doing ts or TOA plot(s)
            elif funcstr == 'W':
                print('  Calculating weighted average')
                ds_load = CalcRegionalAvg(ds_load) if r_domain else CalcWeightedMean(ds_load)
            # Doing AMOC plots:
            elif funcstr == 'M' and attrs[1] != 3:
                print('  Calculating maximum')
                if o_domain:
                    ds_load = ds_load.max('moc_z')
                    ds_load = ds_load.rename({'lat_aux_grid':'lat'})
                else:
                    ds_load.max(('moc_z','lat_aux_grid'))
                
            # If Ensemble mean
            if ens_type and (attrs[1] == 1 or attrs[1] == 2):
                print('  Calculating ensemble mean')
                ds_load = ds_load.mean('ensemble_member')

            ds_load = ds_load.rename(dsname)

            print('  Processing on dataset complete')
            ds_load_list.append(ds_load)

            # Add LENS2 ensemble mean if not 3D variable
            if attrs[1] == 0:
                ds_load_mean = ds_load.mean('slice')
                ds_load_mean = ds_load_mean.rename(dsname+' mean')
                ds_load_list.append(ds_load_mean)
    
    # Merge dataarrays
    ds_proc = xr.merge(ds_load_list)

    print('All datasets merged')

    # Slice time
    ds_proc = ds_proc.loc[slice_time]
    
    return ds_proc

#### CalcWeightedMean

In [16]:
def CalcWeightedMean(ds):
    # Set up
    avg_dim = ('lon','lat')

    # Create weights
    weights = np.cos(np.deg2rad(ds.lat))
    weights.compute()

    # Weight data
    ds_w = ds.weighted(weights)

    # Calculate weighted mean
    ds_mean_w = ds_w.mean(avg_dim, skipna=True)

    ds_mean_w.compute()
    
    return ds_mean_w

#### CalcRegionalAvg

In [17]:
def CalcRegionalAvg(ds):
    # Set up lists
    ds_regavg_list = []
    dom_regname_list = []

    # Cycle over all regions
    for dname, domain in slice_rdom.items():
        da_dom = ds.loc[domain]

        # Calculate weighted average for region
        da_domw = CalcWeightedMean(da_dom)

        # Add to lists
        ds_regavg_list.append(da_domw)
        dom_regname_list.append(dname)

    ds_dom = xr.concat(ds_regavg_list, dim=pd.Index(dom_regname_list, name='region'))
    return ds_dom

#### CalcAnnSeasMean

In [18]:
def CalcAnnSeasMean(ds, atype):
    # atype = 1: annual mean, atype = 2: seasonal mean
    if atype == 1:
        cdim = 'year'
    elif atype == 2:
        cdim = 'season'
    adim = 'time.'+cdim 
    
    # Create weights
    month_length = ds.time.dt.days_in_month
    month_weights = month_length.groupby(adim)/month_length.groupby(adim).sum()

    # Calculate mean for each year
    ds_grp = ds.groupby('time.year')
    month_weights_grp = month_weights.groupby('time.year')

    grp_avg_list = []
        
    for yr, dgrp in ds_grp:
        if atype == 1:
            # Weight data
            dgrp = dgrp.weighted(month_weights_grp[yr])
            
            # Calculate annual mean
            davg = dgrp.mean('time', skipna=True)

        elif atype == 2:
            mw_dgrp = month_weights_grp[yr]
            davg = (dgrp * mw_dgrp).groupby("time.season").sum(dim="time")
            
        davg = davg.expand_dims({'year':[yr]})
        grp_avg_list.append(davg)

    ds_avg = xr.concat(grp_avg_list,dim='year')
    return ds_avg

#### CalcLEMinMax

In [23]:
def CalcLEMinMax(ds):
    for dname, attrs in ds_names.items():
        if attrs[0] and attrs[1] == 0:
            ds[dname+' min'] = ds[dname].min('slice')
            ds[dname+' max'] = ds[dname].max('slice')

#### CalcMonthTrd

In [25]:
def CalcMonthTrd(weighted_avg):
    # Group by month & calculate trends
    grouped = weighted_avg.groupby('time.month')

    ds_list = []
    dsp_list = []
    for mon, dsmon in grouped:
        ds_sizes = dsmon.sizes
        new_time = np.arange(1,ds_sizes['time']+1)
        
        dsmon = dsmon.assign_coords(time=new_time)
        dsmon = dsmon.polyfit(dim='time',deg=1)
        dsmon *= 10
        ds_list.append(dsmon)

    month_trd = xr.concat(ds_list, pd.Index(np.arange(1,13),name='month'))
    
    return month_trd['polyfit_coefficients'].loc[dict(degree=1)]

#### CalcAnnTrd

In [27]:
def CalcAnnTrd(weighted_avg):
    # Calculate annual average
    ann_avg = CalcAnnSeasMean(weighted_avg, 1)

    # Calculate annual trend
    ann_trd = ann_avg.polyfit(dim='year',deg=1)
    ann_trd *= 10
    
    return ann_trd['polyfit_coefficients'].loc[dict(degree=1)]

#### CalcMonAnnTrend

In [29]:
def CalcMonAnnTrend(weighted_avg, time_type):
    # Calculate trends
    trd_list = []
    for dsname, da in weighted_avg.items():
        if time_type == 'm':
            da_trd = CalcMonthTrd(da)
        elif time_type == 'y':
            da_trd = CalcAnnTrd(da)
        trd_list.append(da_trd.rename(dsname))

    ds_trd = xr.merge(trd_list)
            
    return ds_trd

#### CalcXXyrMonAnnTrend

In [31]:
def CalcXXyrMonAnnTrend(ds, time_type):
    
    startyrs = np.arange(t_domain,t_end-(trd_length-2))
    endyrs = np.arange(t_domain+(trd_length-1),t_end+1)
    trd_list = []
    for sy in startyrs:
        print('Calculating trends for '+str(sy)+'-'+str(sy+trd_length-1))
        slice_time_i = dict(time=slice(str(sy)+'-01-01',str(sy+trd_length-1)+'-12-31'))
        ds_sub = copy.deepcopy(ds.loc[slice_time_i])

        dss_trd = CalcMonAnnTrend(ds_sub, time_type)
        trd_list.append(dss_trd)
        
    ds_trd = xr.concat(trd_list, dim=pd.Index(startyrs, name='start_year'))
    ds_trd = ds_trd.assign_coords(end_year=('start_year', endyrs))

    return ds_trd

#### CalcGridArea

In [35]:
def CalcGridArea(ds):
    # For ERA5 0.25x0.25 grid
    dlat = 0.25
    dlon = 0.25
    R = 6367.47 # km

    lons, lats = np.meshgrid(ds.lon.values, ds.lat.values)

    dy = R*np.deg2rad(dlat)
    dx = np.deg2rad(dlon)*R*np.cos(np.deg2rad(lats))

    area = np.abs(dx*dy)
    ds = ds.assign_coords(area=(('lat','lon'), area))

    return ds

#### CalcSIE

In [36]:
def CalcSIE(ds, set_type):
    # Extracts aice variable and only counts cells with sic > 15%
    ds_aice = ds
    ds_aice = xr.where(ds_aice > .15,1,0)

    # Multiples selected sic cells with tarea, then only selects Arctic sea ice, sums over the entire domain, then converts to km2
    if set_type == 3:
        dsa = (ds_aice*ds['area']).sum(dim=['lon','lat'])*1e-6
    else:
        dsa = (ds_aice*ds['tarea']).sum(dim=['ni','nj'])*1e-6*1e-6

    # Modifies attributes of DataArray accordingly
    dsa.attrs['units'] = 'million km^2'
    dsa.attrs['long_name'] = 'Arctic sea ice area'

    # Assigns new variable to dataset and returns original dataset
    return dsa

#### CalcTrend

In [37]:
def CalcTrend(da, time_type):
    da_b = da.dropna(dim='time')
    slope, intercept, rval, pval, stderr = xr.apply_ufunc(stats.linregress,
                                                          da_b[time_type], da_b,
                                                          input_core_dims=[[time_type], [time_type]],
                                                          output_core_dims=[[],[],[],[],[]],
                                                          output_dtypes=[xr.core.dataarray.DataArray,xr.core.dataarray.DataArray,xr.core.dataarray.DataArray,
                                                                        xr.core.dataarray.DataArray,xr.core.dataarray.DataArray],
                                                          vectorize=True,
                                                          dask='parallelized',
                                                          dask_gufunc_kwargs=dict(allow_rechunk=True))
    slope = slope*10
    return slope, pval

#### AddCoordTrend

In [38]:
def AddCoordTrend(da, time_type):
    da_sizes = da.sizes
    new_time = np.arange(1,da_sizes[time_type]+1)
    if time_type == 'time':
        da = da.assign_coords(time=new_time)
    elif time_type == 'year':
        da = da.assign_coords(year=new_time)
    return da

#### SaveData

In [46]:
def SaveData(ds, plot_type, varname, tavg, plot_level=None):
    level_str = '' if plot_level == None else 'Z'+str(int(plot_level))+'.'
    domain_str = sd_str
    if r_domain:
        domain_str += '.'+rd_str
    if o_domain:
        domain_str += '.'+od_str
    
    filename = plot_type+'.'+varname+'.'+domain_str+'.'+level_str+td_str+'.'+tavg+'.'+ens_str+'.nc'
    
    print('Saving '+filename)
    # File format is:
    # (plot type, including special averaging, i.e. anomalies).varname.spatialdomain.timedomain.timeaveraging.ensemble_type.nc
    ds.to_netcdf(path_to_plotdata+filename, format='NETCDF4')
    
    return None

### Process and load data

In [47]:
%%time

ds_proc = CreateMasterDS(var)

Figure out processing calculations to do
LENS2 piControl
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
LENS2 HIST-SSP370
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
GHG2
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
PiC_UVnudge_LM2006
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
HIST_UVnudge_LM
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
GHG_UVnudge_LM
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
RAPID
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Processing on dataset complete
All datasets merge

### Plotting set-up

In [49]:
%%time

mon_str = np.array(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
seas_str = np.array(['MAM','JJA','SON','DJF'])


CPU times: user 27 µs, sys: 4 µs, total: 31 µs
Wall time: 32.2 µs


## Month trend plots
### Set up

In [50]:
if plot_types['mtrd'][0]:
    graph_type_str = 'Linear.Trend'
    ## Select plot type - timeseries or monthly - to make and assign variables accordings
    

    # Timeseries
    if time_avg == 4:
        dim_avg = 'time.month'
        period = 'month'
        trdyr_str = '.'+str(trd_length)+'yr'

### Data processing

In [ ]:
%%time

if plot_types['mtrd'][0]:
    
    if subset_time:
        # Calculate monthly trends
        ds_mon_trd = CalcMonAnnTrend(ds_proc, 'm')
    
        # Calculate annual trends
        ds_ann_trd = CalcMonAnnTrend(ds_proc, 'y')
    
        # Write out data
        SaveData(ds_mon_trd, graph_type_str, var, 'month')
        SaveData(ds_ann_trd, graph_type_str, var, 'year')

    else:
        if var != 'MOC':
            ds_mon_trd = CalcXXyrMonAnnTrend(ds_proc, 'm')
                
            SaveData(ds_mon_trd, graph_type_str+trdyr_str, var, 'month')
            
        ds_ann_trd = CalcXXyrMonAnnTrend(ds_proc, 'y')
            
        SaveData(ds_ann_trd, graph_type_str+trdyr_str, var, 'year')

Calculating trends for 1950-1969
Calculating trends for 1951-1970
Calculating trends for 1952-1971
Calculating trends for 1953-1972
Calculating trends for 1954-1973
Calculating trends for 1955-1974
Calculating trends for 1956-1975
Calculating trends for 1957-1976
Calculating trends for 1958-1977
Calculating trends for 1959-1978
Calculating trends for 1960-1979
Calculating trends for 1961-1980
Calculating trends for 1962-1981
Calculating trends for 1963-1982
Calculating trends for 1964-1983
Calculating trends for 1965-1984
Calculating trends for 1966-1985
Calculating trends for 1967-1986
Calculating trends for 1968-1987
Calculating trends for 1969-1988
Calculating trends for 1970-1989
Calculating trends for 1971-1990
Calculating trends for 1972-1991
Calculating trends for 1973-1992
Calculating trends for 1974-1993
Calculating trends for 1975-1994
Calculating trends for 1976-1995
Calculating trends for 1977-1996
Calculating trends for 1978-1997
Calculating trends for 1979-1998
Calculatin

## Line plots
### Set up

In [52]:
if plot_types['line'][0]:
    graph_type_str = 'Linear'
        

    # Time averaging for yearly plot
    if time_avg == 1:
        dim_avg = 'time.year'  
        period='year'
    if time_avg == 2:
        dim_avg = 'time.season'
        period = 'season'
    if time_avg == 4:
        dim_avg = 'time.month'
        period ='time'

### Data processing

In [53]:
%%time

if plot_types['line'][0]:

    ## Absolute (only one that will be used for SIA-one month, TOA, AMOC)
    # Yearly averaging for absolute
    if time_avg == 1:
        ds_abs = CalcAnnSeasMean(ds_proc, 1)
        CalcLEMinMax(ds_abs)
        
        SaveData(ds_abs, graph_type_str+'.abs', var, period)

    elif time_avg == 2:
        ds_abs = CalcAnnSeasMean(ds_proc, 2)
        CalcLEMinMax(ds_abs)
        
        SaveData(ds_abs, graph_type_str+'.abs', var, period)

    elif time_avg == 4:
        # If only plotting one month from SIA data
        ds_abs = ds_proc.groupby(dim_avg)

        for m, ds in ds_abs:
            CalcLEMinMax(ds)

            SaveData(ds, graph_type_str+'.abs', var, mon_str[m-1])

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 7.39 µs


In [ ]:
client.shutdown()